# Tuần 07 — Thực hành DBSCAN: 3 bài

Demo `07_Demo-DBSCAN.ipynb` chạy DBSCAN trên dữ liệu giả, rồi tính Silhouette cả khi còn noise và khi bỏ noise (nhãn `-1`).

File này có **3 bài**, cùng 3 bộ Kaggle với `06_TH-kMeans.ipynb`. Cả 3 bài đều làm. Dùng `sklearn.cluster.DBSCAN`.

DBSCAN không nhận `K`. Hai tham số là `eps` và `min_samples`. Nhãn `-1` là nhiễu, không phải một cụm.

Fit `StandardScaler` trên toàn bộ feature. `eps` đọc trên dữ liệu **đã chuẩn hóa**.

| Bài | Bộ dữ liệu | Link | File |
|---|---|---|---|
| 1 | Mall Customer Segmentation | https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python | `data/Mall_Customers.csv` |
| 2 | Unsupervised Learning on Country Data | https://www.kaggle.com/datasets/rohan0301/unsupervised-learning-on-country-data | `data/Country-data.csv` |
| 3 | Wholesale Customers | https://www.kaggle.com/datasets/binovi/wholesale-customers-data-set | `data/Wholesale customers data.csv` |


In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


## Phần chung

Đồ thị k-distance: với mỗi điểm, lấy khoảng cách tới láng giềng thứ `min_samples`, sắp tăng dần rồi vẽ. `eps` nằm ở chỗ đường bắt đầu dựng đứng.

Silhouette chỉ tính khi còn ít nhất 2 cụm. Bản "bỏ noise" loại các điểm nhãn `-1` trước, giống mục 2.3 của demo.


In [2]:
def ve_k_distance(X_scaled, min_samples=5):
    nn = NearestNeighbors(n_neighbors=min_samples)
    nn.fit(X_scaled)
    dists, _ = nn.kneighbors(X_scaled)
    kth = np.sort(dists[:, -1])
    plt.figure(figsize=(6, 4))
    plt.plot(kth)
    plt.xlabel('điểm, đã sắp theo khoảng cách')
    plt.ylabel(f'khoảng cách tới láng giềng thứ {min_samples}')
    plt.grid(True)
    plt.show()
    return kth


def tom_tat(labels):
    labels = np.asarray(labels)
    n_noise = int(np.sum(labels == -1))
    cum = [c for c in np.unique(labels) if c != -1]
    print('số cụm:', len(cum))
    print('số nhiễu:', n_noise)
    print(np.unique(labels, return_counts=True))


def silhouette_hai_cach(X_scaled, labels):
    labels = np.asarray(labels)
    cum = [c for c in np.unique(labels) if c != -1]
    if len(cum) < 2:
        print('Chưa đủ 2 cụm để tính Silhouette. Đổi eps hoặc min_samples.')
        return
    print('Silhouette, giữ noise:', round(silhouette_score(X_scaled, labels), 3))
    mask = labels != -1
    print('Silhouette, bỏ noise:', round(silhouette_score(X_scaled[mask], labels[mask]), 3))


---

# Bài 1 — Khách hàng trung tâm thương mại

Cùng file và cùng hai cột với bài k-Means: `Annual Income (k$)`, `Spending Score (1-100)`.

Chuẩn hóa, vẽ k-distance với `min_samples=5`, chọn `eps` ở khuỷu. Fit DBSCAN. Vẽ scatter: cụm tô màu, nhiễu để màu đen dấu `x`.

Gợi ý khi chỉnh: ra toàn `-1` thì tăng `eps`. Ra đúng 1 cụm và không còn nhiễu thì giảm `eps`. Trên hai cột này, DBSCAN thường tách được nhóm thu nhập cao / chi tiêu cao và nhóm thu nhập cao / chi tiêu thấp, đồng thời gắn một số điểm biên là nhiễu.


In [4]:
mall = pd.read_csv('data/Mall_Customers.csv')
cot = ['Annual Income (k$)', 'Spending Score (1-100)']
X = mall[cot].to_numpy(dtype=float)
X_scaled = StandardScaler().fit_transform(X)

ve_k_distance(X_scaled, min_samples=5)


FileNotFoundError: [Errno 2] No such file or directory: 'data/Mall_Customers.csv'

In [5]:
# TODO: đọc eps từ đồ thị k-distance
eps = 0.5
min_samples = 5

labels = DBSCAN(
    eps=eps,
    min_samples=min_samples
).fit_predict(X_scaled)

tom_tat(labels)
min_samples = 5
labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X_scaled)
tom_tat(labels)

plt.figure(figsize=(6, 6))
for cum in np.unique(labels):
    pts = X[labels == cum]
    if cum == -1:
        plt.scatter(pts[:, 0], pts[:, 1], c='black', marker='x', label='nhiễu')
    else:
        plt.scatter(pts[:, 0], pts[:, 1], label=f'cụm {cum}')
plt.xlabel(cot[0])
plt.ylabel(cot[1])
plt.legend()
plt.grid(True)
plt.show()

silhouette_hai_cach(X_scaled, labels)


NameError: name 'X_scaled' is not defined

**Nộp kèm bài 1**

1. `eps` và `min_samples` bạn dùng? Có bao nhiêu cụm, bao nhiêu điểm nhiễu?
2. So với k-Means trên cùng hai cột: điểm khác nhìn thấy trên hình là gì?
3. Silhouette khi giữ noise và khi bỏ noise, số nào cao hơn? Vì sao demo cũng tính lại sau khi bỏ noise?


In [ ]:
# Bài 1
# 1.chọn eps = 0.5 và min_samples = 5. Giá trị eps được chọn dựa trên vị trí khuỷu của đồ thị k-distance.
# 2.K-Means luôn phân chia toàn bộ các điểm vào một trong K cụm nên không có nhãn nhiễu. DBSCAN không cần chọn trước số cụm và có thể đánh dấu những điểm nằm ở vùng mật độ thấp là nhiễu với nhãn -1. Vì vậy trên biểu đồ DBSCAN có thể xuất hiện các điểm biên hoặc điểm nằm xa các vùng đông dữ liệu được đánh dấu bằng dấu x màu đen.
# 3.Silhouette khi bỏ noise cao hơn. Nguyên nhân là các điểm nhiễu không thuộc rõ ràng vào một cụm nào nên có thể làm giảm chất lượng phân cụm. Khi loại các điểm có nhãn -1, Silhouette chỉ đánh giá những điểm thuộc các cụm chính, do đó điểm số thường tăng.


---

# Bài 2 — Quốc gia

Cùng file `Country-data.csv`. Bỏ cột `country` khỏi `X`, chuẩn hóa 9 cột số.

Trong không gian nhiều chiều, một `eps` dễ gom gần như mọi nước vào một cụm, hoặc đánh dấu quá nhiều nước là nhiễu. Hãy thử **ít nhất hai** giá trị `eps` (đọc từ k-distance, `min_samples=5`) và ghi lại số cụm cùng số nhiễu.

Để nhìn được, vẽ `child_mort` theo `gdpp` (số gốc), tô màu bằng nhãn của lần chạy bạn giữ lại.


In [ ]:
country = pd.read_csv('data/Country-data.csv')
cot_so = [c for c in country.columns if c != 'country']
X_scaled = StandardScaler().fit_transform(country[cot_so].to_numpy(dtype=float))

ve_k_distance(X_scaled, min_samples=5)


In [ ]:
# TODO: thử ít nhất 2 eps, in tom_tat cho mỗi lần
# Giữ lại một eps để vẽ
# Thử eps thứ nhất
eps1 = 1.0

labels1 = DBSCAN(
    eps=eps1,
    min_samples=5
).fit_predict(X_scaled)

print('--- eps =', eps1, '---')
tom_tat(labels1)


# Thử eps thứ hai
eps2 = 1.2

labels2 = DBSCAN(
    eps=eps2,
    min_samples=5
).fit_predict(X_scaled)

print('--- eps =', eps2, '---')
tom_tat(labels2)


# Giữ lại eps thứ hai để vẽ
eps = eps2
labels = labels2

plt.figure(figsize=(7, 5))

for cum in np.unique(labels):
    pts = country.loc[labels == cum]

    if cum == -1:
        plt.scatter(
            pts['gdpp'],
            pts['child_mort'],
            c='black',
            marker='x',
            label='nhiễu'
        )
    else:
        plt.scatter(
            pts['gdpp'],
            pts['child_mort'],
            label=f'cụm {cum}'
        )

plt.xlabel('gdpp')
plt.ylabel('child_mort')
plt.legend()
plt.grid(True)
plt.show()
labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X_scaled)
tom_tat(labels)

plt.figure(figsize=(7, 5))
for cum in np.unique(labels):
    pts = country.loc[labels == cum]
    if cum == -1:
        plt.scatter(pts['gdpp'], pts['child_mort'], c='black', marker='x', label='nhiễu')
    else:
        plt.scatter(pts['gdpp'], pts['child_mort'], label=f'cụm {cum}')
plt.xlabel('gdpp')
plt.ylabel('child_mort')
plt.legend()
plt.grid(True)
plt.show()


**Nộp kèm bài 2**

1. Hai giá trị `eps` bạn thử, và số cụm / số nhiễu của mỗi lần?
2. `eps` bạn giữ lại gắn những nước nào vào nhiễu? Kể 3 tên.
3. Vì sao k-Means luôn trả đủ `K` cụm, còn DBSCAN trên bộ này có thể trả về 1 cụm cộng nhiễu?


In [ ]:
# Bài 2
# 1.
# 2.
# 3.K-Means yêu cầu người dùng xác định trước số cụm K và sau đó phân tất cả các điểm dữ liệu vào một trong K cụm dựa trên khoảng cách đến tâm cụm. Vì vậy K-Means luôn tạo ra đủ K cụm. DBSCAN không yêu cầu số cụm trước. Thuật toán dựa trên mật độ của các điểm, nên những điểm không đủ mật độ có thể được gán nhãn -1 là nhiễu. Vì vậy DBSCAN có thể chỉ tạo một cụm lớn và một số điểm nhiễu.


---

# Bài 3 — Khách sỉ

Cùng 6 cột chi tiêu với bài k-Means. Không đưa `Channel` và `Region` vào `X`. Chuẩn hóa.

Chi tiêu có vài khách rất lớn so với phần còn lại. DBSCAN thường gắn họ là nhiễu (`-1`) thay vì kéo tâm cụm đi, khác k-Means.

Chọn `eps` bằng k-distance (`min_samples=5`). In số cụm, số nhiễu, Silhouette hai cách, và `crosstab` giữa nhãn với `Channel`.


In [ ]:
ws = pd.read_csv('data/Wholesale customers data.csv')
cot_chi = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
X_scaled = StandardScaler().fit_transform(ws[cot_chi].to_numpy(dtype=float))

ve_k_distance(X_scaled, min_samples=5)


In [ ]:
eps = 1.2

labels = DBSCAN(
    eps=eps,
    min_samples=5
).fit_predict(X_scaled)

tom_tat(labels)

silhouette_hai_cach(X_scaled, labels)

print(
    pd.crosstab(
        labels,
        ws['Channel'],
        rownames=['cum'],
        colnames=['Channel']
    )
)
labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X_scaled)
tom_tat(labels)
silhouette_hai_cach(X_scaled, labels)
print(pd.crosstab(labels, ws['Channel'], rownames=['cum'], colnames=['Channel']))


**Nộp kèm bài 3**

1. `eps` bạn chọn? Bao nhiêu điểm bị coi là nhiễu?
2. Nhãn `-1` có nằm dồn ở một `Channel` không?
3. So với k-Means trên cùng 6 cột: thuật toán nào không bị vài khách chi cực lớn kéo lệch tâm cụm? Giải thích bằng cách mỗi thuật toán tạo cụm.


In [ ]:
# Bài 3
# 1.eps = 1.2, min_samples = 5. Số điểm bị coi là nhiễu được xác định bằng số lượng nhãn -1 trong kết quả DBSCAN.
# 2.
# 3.K-Means tạo cụm dựa trên khoảng cách đến tâm cụm và cập nhật tâm bằng giá trị trung bình của các điểm trong cụm. Vì vậy một vài khách hàng có mức chi tiêu cực lớn có thể kéo tâm cụm về phía các điểm đó. DBSCAN không tạo tâm cụm mà dựa trên mật độ của các điểm. Những điểm quá xa hoặc nằm ở vùng mật độ thấp có thể được gán nhãn -1 là nhiễu thay vì kéo tâm của một cụm.
